# 02 · How differently do products sell?

I want to know what hides behind the store average. Is a zero unusual? Does an apparent calendar effect survive a fairer comparison? How much do our regime labels depend on where we start the clock? Each section asks a question, tries a comparison, and explains what we decided for the next stage.

Every sales result here uses training. The later phases reuse the primary label computed from the full training block. See the [mathematical definitions](../docs/TECHNICAL_BLUEPRINT.md).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "configs" / "config.yaml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the project root or its notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_audit import load_config

config_path = project_root / "configs" / "config.yaml"
config = load_config(config_path)
processed = project_root / config["data"]["processed_dir"]

import json
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from src.regimes import classify, run_regimes

partitions = [processed / "panel" / f"{store}.parquet" for store in config["data"]["stores"]]
if missing := [path.name for path in partitions if not path.is_file()]:
    raise FileNotFoundError(f"Run notebook 01 first. Missing: {', '.join(missing)}")

regime_dir = project_root / config["regimes"]["output_dir"]
run_regimes(config_path)
profile = json.loads((regime_dir / "summary.json").read_text(encoding="utf-8"))
series = pd.read_parquet(regime_dir / "sku_regimes.parquet")
counts = series.groupby(["store_id", "regime"], observed=True).size().unstack(fill_value=0)
counts = counts.reindex(columns=["Stable", "Volatile", "Intermittent"], fill_value=0)
display(counts.assign(total=counts.sum(axis=1)))
print(f"Training: {profile['training_start']} through {profile['training_end']}")
print(f"Training item-days: {profile['item_days']:,}")
print(f"Zero-sales share: {profile['zero_share']:.1%}")
print(f"Series with insufficient positive history: {profile['insufficient_history_series']:,}")

## 1. Start with the whole store, then look at individual items

The summaries above cover all configured stores. The detailed questions below use the first configured store, CA_1 by default. Everything in this notebook comes from training. I want to see whether the average day is representative, and whether the same amount of average sales can come from very different buying patterns.

In [ ]:
import numpy as np

focus_store = config["data"]["stores"][0]
start, end = pd.Timestamp(profile["training_start"]), pd.Timestamp(profile["training_end"])
selected = series.loc[series["store_id"].eq(focus_store)].copy()
train = pd.read_parquet(
    processed / "panel" / f"{focus_store}.parquet",
    columns=["store_id", "item_id", "cat_id", "dept_id", "date", "weekday", "snap", "has_event", "sales", "sell_price", "week_end"],
    filters=[("date", ">=", start), ("date", "<=", end)],
).sort_values(["item_id", "date"], ignore_index=True)
assert train["date"].between(start, end).all()
units_by_item = train.groupby("item_id")["sales"].sum().sort_values(ascending=False)
concentration = units_by_item.cumsum() / units_by_item.sum()
top_count = max(1, int(np.ceil(len(units_by_item) * 0.10)))
print(f"{focus_store}: the highest-volume 10% of items account for "
      f"{concentration.iloc[top_count - 1]:.1%} of recorded training units.")
print(f"The median item sold {units_by_item.median():,.0f} units over training.")
display(train["sales"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame("all item-days")
        .join(train.loc[train["sales"].gt(0), "sales"]
              .describe(percentiles=[0.5, 0.9, 0.95, 0.99]).rename("positive item-days")).round(2))

fig, (time_ax, mix_ax) = plt.subplots(1, 2, figsize=(11, 3.7), layout="constrained")
daily = train.groupby("date")["sales"].sum()
time_ax.plot(daily, color="#819796", linewidth=0.7, alpha=0.75, label="Daily units")
time_ax.plot(daily.rolling(28, min_periods=28).mean(), color="#113B4B", label="28-day mean")
time_ax.set(title=f"{focus_store} sales through training", ylabel="Units per day")
time_ax.legend()
mix_ax.plot(np.arange(1, len(concentration) + 1) / len(concentration), concentration, color="#113B4B")
mix_ax.plot([0, 1], [0, 1], ":", color="#819796")
mix_ax.set(title="Sales concentrated by SKU", xlabel="Share of SKUs, highest volume first", ylabel="Share of units")
mix_ax.xaxis.set_major_formatter(PercentFormatter(1))
mix_ax.yaxis.set_major_formatter(PercentFormatter(1))
plt.show()

**Why this matters.** A high-volume SKU matters more when counting total shortage units. It should not silently dominate a claim about how well a method works for the typical SKU. Later comparisons will show where volume and per-series performance tell different stories.

### Bring back the proposal's positive-sales distribution

The earlier notebook separated zero days from the distribution of positive units. Combining them in one histogram can hide both the mass at zero and the long tail. Compare the two views, then ask whether trends and category averages conceal differences among departments.

In [ ]:
positive_units = train.loc[train["sales"].gt(0), "sales"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
zero_counts = train["sales"].eq(0).value_counts().reindex([True, False])
axes[0].bar(["Zero", "Positive"], zero_counts, color=["#789695", "#19647E"])
axes[0].set(title="How often was a sale recorded?", ylabel="Training item-days")
positive_units.value_counts().sort_index().plot(ax=axes[1], color="#19647E", logy=True)
axes[1].set(title="How large were positive sales?", xlabel="Units on a positive day", ylabel="Item-days (log scale)")
plt.show()

In [ ]:
months = daily.groupby(daily.index.to_period("M")).agg(["mean", "size"])
months["complete"] = months["size"].eq(months.index.days_in_month)
departments = train.groupby(["cat_id", "dept_id"], observed=True).agg(
    series=("item_id", "nunique"), mean_units=("sales", "mean"), units=("sales", "sum"),
)
departments["zero_share"] = (
    train["sales"].eq(0).groupby([train["cat_id"], train["dept_id"]], observed=True).mean()
)
display(departments.sort_values("mean_units", ascending=False).round(3))
display(months.loc[~months["complete"]].rename(columns={"size": "days observed", "mean": "mean daily units"}).round(1))
print("Dates with zero recorded units across the store:", ", ".join(daily.loc[daily.eq(0)].index.strftime("%Y-%m-%d")) or "none")

fig, (day_ax, month_ax) = plt.subplots(1, 2, figsize=(11, 3.7), layout="constrained")
day_ax.plot(daily.rolling(7, min_periods=7).mean(), color="#19647E")
day_ax.set(title="Trailing seven-day store mean", ylabel="Units per day")
month_dates = months.index.to_timestamp()
month_ax.plot(month_dates, months["mean"].where(months["complete"]), color="#19647E", marker="o", label="Complete month")
partial = ~months["complete"]
month_ax.scatter(month_dates[partial], months.loc[partial, "mean"], color="#b75f25", marker="x", label="Partial month")
month_ax.set(title="Mean daily store units by month", ylabel="Units per day")
month_ax.legend()
plt.show()

The training boundary can leave a partial month. Its mean uses fewer days and a possibly different weekday mix. A store-total zero is also unusual, but the files do not identify whether the store was closed. Keep these dates in the primary analysis and interpret the aggregates with those limits.

### How do categories and calendar contexts differ?

An **item-day** is one SKU observed on one date in one store. Mean units per item-day makes groups of different sizes comparable. Positive-day skewness measures asymmetry after zeros are excluded. Raw skewness also reflects the zero mass.

The M5 SNAP flag refers to the U.S. Supplemental Nutrition Assistance Program. An event day has a named calendar event in M5. The associations below may reflect differences in weekday, season, and product mix.

In [ ]:
by_store = pd.read_csv(regime_dir / "eda_by_store.csv")
by_category = pd.read_csv(regime_dir / "eda_by_category.csv")
by_weekday = pd.read_csv(regime_dir / "eda_by_weekday.csv")
by_context = pd.read_csv(regime_dir / "eda_by_context.csv")
columns = ["item_days", "mean_units", "zero_share", "sales_skewness", "positive_skewness"]
display(by_store[["store_id", *columns]].round(3))
display(by_category[["store_id", "cat_id", *columns]].round(3))
display(by_context[["store_id", "state_id", "snap", "has_event", *columns]].round(3))

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, (weekday_ax, zeros_ax) = plt.subplots(
    1, 2, figsize=(11, max(4, len(by_category) / 4)), layout="constrained"
)
for store, rows in by_weekday.groupby("store_id", sort=True):
    ordered = rows.sort_values("weekday")
    weekday_ax.plot(ordered["weekday"], ordered["mean_units"], marker="o", label=store)
weekday_ax.set(title="Sales by weekday", ylabel="Mean units per item-day", xlabel="Weekday")
weekday_ax.set_xticks(range(7), days)
weekday_ax.legend()

category_plot = by_category.sort_values("zero_share")
labels = category_plot["store_id"] + " · " + category_plot["cat_id"]
zeros_ax.barh(labels, category_plot["zero_share"], color="#19647E")
zeros_ax.set(title="Zero-sales share by category", xlabel="Share of training item-days", xlim=(0, 1))
zeros_ax.xaxis.set_major_formatter(PercentFormatter(1))
plt.show()

### Does the calendar signal survive a fairer comparison?

A raw average can make SNAP look predictive simply because flagged dates coincide with busier weekdays or months. I compare flagged and unflagged days within the same year-month and weekday, and for SNAP use non-event days. For events I also match SNAP status. Each eligible group contributes one comparison. Groups with no comparable days are excluded and counted. This is a descriptive check on aggregated store units. Exact event names, product mix and other changes can still matter.

In [ ]:
date_totals = train.groupby(["date", "weekday", "snap", "has_event"], as_index=False)["sales"].sum()
date_totals["month"] = date_totals["date"].dt.to_period("M")
calendar_rows = []
for flag, samples, strata in [
    ("snap", date_totals.loc[~date_totals["has_event"]], ["month", "weekday"]),
    ("has_event", date_totals, ["month", "weekday", "snap"]),
]:
    by_flag = samples.groupby(flag)["sales"].mean().reindex([False, True])
    cells = samples.groupby([*strata, flag])["sales"].mean().unstack(flag).reindex(columns=[False, True])
    pairs = cells.dropna()
    calendar_rows.append({
        "flag": flag,
        "raw difference, units/day": by_flag.loc[True] - by_flag.loc[False],
        "matched mean difference, units/day": (pairs[True] - pairs[False]).mean(),
        "matched groups": len(pairs),
        "groups excluded": len(cells) - len(pairs),
    })
display(pd.DataFrame(calendar_rows).round(1))

### Does sales volume move when prices change?

Comparing expensive items with cheap ones mixes fundamentally different products. After the plots below, compare each item's sales with its own preceding week. Use adjacent, complete training weeks with an observed listed price. Look at price increases, decreases and unchanged prices separately. Changes in promotions, demand, and availability can coincide with prices, so the sign is evidence about association only. The comparison also leaves out weeks without a listed price.

In [ ]:
fig, (distribution, over_time) = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
price_weeks = train[["item_id", "week_end", "sell_price"]].drop_duplicates(["item_id", "week_end"])
price_weeks["sell_price"].dropna().plot.hist(ax=distribution, bins=40, color="#19647E")
distribution.set(title="One observation per listed SKU-week", xlabel="Listed weekly price (USD)", ylabel="SKU-weeks")
missing_per_day = train["sell_price"].isna().groupby(train["date"]).mean()
over_time.plot(missing_per_day, color="#b75f25")
over_time.yaxis.set_major_formatter(PercentFormatter(1))
over_time.set(title="Missing weekly prices through training", ylabel="Share of SKUs")
plt.show()

top_item = units_by_item.index[0]
item = train.loc[train["item_id"].eq(top_item)].set_index("date")
fig, (unit_axis, price_axis) = plt.subplots(2, 1, figsize=(10, 4), sharex=True, layout="constrained")
unit_axis.plot(item["sales"], color="#19647E")
price_axis.step(item.index, item["sell_price"], where="post", color="#b75f25")
unit_axis.set(title=f"Highest-volume training item · {top_item}", ylabel="Daily units")
price_axis.set(ylabel="Listed price (USD)")
plt.show()

The price histogram counts one entry per SKU-week. Repeating each weekly price on seven rows would overweight longer observed weeks. The top-selling item is chosen by a reproducible rule, but its price and sales pattern cannot represent the full catalog. A single pooled price-sales correlation also mixes items with very different typical prices and volumes. The adjacent-week comparison below is one alternative and still leaves availability and promotion confounding.

In [ ]:
weekly = (
    train.groupby(["item_id", "week_end"], as_index=False)
    .agg(units=("sales", "sum"), days=("sales", "size"), price=("sell_price", "first"))
    .sort_values(["item_id", "week_end"])
)
previous = weekly.groupby("item_id")[["week_end", "units", "days", "price"]].shift()
valid_pairs = (
    (weekly["week_end"] - previous["week_end"]).dt.days.eq(7)
    & weekly["days"].eq(7) & previous["days"].eq(7)
    & weekly["price"].notna() & previous["price"].notna()
)
change = (weekly.loc[valid_pairs, "price"] - previous.loc[valid_pairs, "price"]).round(2)
price_comparison = pd.DataFrame({
    "direction": np.select([change.lt(0), change.gt(0)], ["Lower", "Higher"], default="Unchanged"),
    "weekly unit change": weekly.loc[valid_pairs, "units"] - previous.loc[valid_pairs, "units"],
})
display(price_comparison.groupby("direction")["weekly unit change"].agg(["size", "mean", "median"]).round(2))

**What I learned from the default CA_1 run.** The SNAP association stays positive after matching. The broad event flag stays negative, but only 36 matched groups remain. An average across all events could hide effects for individual holidays. Sales also rose on average in both price-rise and price-drop weeks, which makes me skeptical of any elasticity story from this table alone.

**Feature decision.** Keep weekday, scheduled SNAP/events, and a price feature built from information eligible at the origin. Test whether adding those features improves forecasts on development dates. A training association can motivate a feature, but only a fair comparison shows whether the feature helps prediction.

## 2. Test how the regime rules behave

The active clock starts at the first positive training sale. Active zero share counts the fraction of active dates with zero sales. Positive-day $\mathrm{CV}^2$ divides the sample variance of positive daily sales by the squared positive-day mean. The first sale handles long leading zero runs consistently. The exact listing date remains unknown.

1. **Intermittent** when positive history is too short or active zero share meets its cutoff.
2. **Volatile** when the SKU passes the first rule and positive-day $\mathrm{CV}^2$ meets its cutoff.
3. **Stable** otherwise.

Vary both cutoffs on the same training data. Large count changes near the chosen settings indicate sensitivity in later regime-level results.

In [ ]:
settings = config["regimes"]
print(f"Active zero share ≥ {settings['zero_share_threshold']:.0%}")
print(f"Positive-day CV² ≥ {settings['cv2_threshold']:.2f}")
print(f"Minimum positive dates = {settings['min_positive_days']}")
examples = series.sort_values(["store_id", "item_id"]).groupby("regime", observed=True).head(2)
display(examples[[
    "store_id", "item_id", "regime", "first_positive_date", "positive_days",
    "full_zero_share", "active_zero_share", "positive_cv2", "insufficient_history",
]].round({"full_zero_share": 3, "active_zero_share": 3, "positive_cv2": 3}))

thresholds = pd.read_csv(regime_dir / "threshold_sensitivity.csv")
sensitivity = thresholds.pivot_table(
    index=["store_id", "zero_share_threshold", "cv2_threshold"],
    columns="regime", values="sku_count", fill_value=0, observed=True,
)
display(sensitivity.reindex(columns=counts.columns, fill_value=0))

### What happens if we count leading zeros as intermittency?

One plausible first attempt is to call any SKU with a full-training zero share of at least 60% intermittent. Some items have long stretches with no recorded sales at the beginning of training. We will swap in the full-window zero share while keeping the other rules fixed, then count who moves.

The 0.60 cutoff is an exploratory convention. For a history with sales on some days and zeros on 60%, the average interval is about $1/(1-0.60)=2.5$ days per positive day. A $\mathrm{CV}^2$ cutoff of 0.49 corresponds to a positive-day standard deviation of 70% of the positive-day mean. The grid above tests nearby choices. These labels describe recorded-sales patterns and the cutoffs have not been optimized for inventory outcomes.

In [ ]:
full_window_rule = np.select(
    [
        selected["insufficient_history"] | selected["full_zero_share"].ge(settings["zero_share_threshold"]),
        selected["positive_cv2"].ge(settings["cv2_threshold"]),
    ],
    ["Intermittent", "Volatile"],
    default="Stable",
)
transition = pd.crosstab(
    selected["regime"], pd.Series(full_window_rule, index=selected.index),
    rownames=["Active-window rule"], colnames=["Full-training zero share"],
)
display(transition)
print(f"Changing only the zero-share clock would relabel "
      f"{int(selected['regime'].ne(full_window_rule).sum()):,} of {len(selected):,} SKUs.")

midpoint = start + pd.Timedelta(days=(end - start).days // 2)
options = {key: settings[key] for key in ("zero_share_threshold", "cv2_threshold", "min_positive_days")}
first_half = classify(train, start=start, end=midpoint, **options).set_index("item_id")
second_half = classify(train, start=midpoint + pd.Timedelta(days=1), end=end, **options).set_index("item_id")
both_observed = ~first_half["insufficient_history"] & ~second_half["insufficient_history"]
display(pd.crosstab(first_half["regime"], second_half["regime"],
                    rownames=["Earlier training half"], colnames=["Later training half"]))
stability = first_half.loc[both_observed, "regime"].eq(second_half.loc[both_observed, "regime"])
print(f"Among SKUs with sufficient history in both halves, {stability.sum():,} of "
      f"{len(stability):,} retained their label.")

**Regime decision.** Use the active-history rule for the primary labels and keep full-window zero share for context. The alternative exposes the influence of leading zeros. The comparison across halves mixes real changes with sensitivity to shorter samples, so use it to question how permanent the labels are. Items with insufficient positives remain flagged. An item can sell regularly and still vary greatly in quantity; that is why the volatility rule uses positive-day sizes separately.

### What do the labels look like on real items?

Pick the middle-ranked item by training volume within each regime, breaking ties by item ID. This gives a reproducible example for each regime. Individual examples illustrate patterns and do not establish the behavior of every item in a group. Compare the observed daily series with totals over two and eight days, the two protection periods planned for the simulator. Windows overlap and stay inside training.

In [ ]:
ranked = (
    selected.assign(training_units=selected["item_id"].map(units_by_item))
    .sort_values(["regime", "training_units", "item_id"])
)
representatives = pd.DataFrame(
    [items.iloc[len(items) // 2] for _, items in ranked.groupby("regime", sort=False)]
)
fig, axes = plt.subplots(len(representatives), 1, figsize=(11, 2.15 * len(representatives)),
                         sharex=True, squeeze=False, layout="constrained")
protection_rows = []
for axis, (_, item) in zip(axes[:, 0], representatives.iterrows(), strict=True):
    sales = train.loc[train["item_id"].eq(item["item_id"])].set_index("date")["sales"]
    axis.vlines(sales.index, 0, sales, color="#19647E", linewidth=0.8)
    axis.set(title=f"{item['regime']} · {item['item_id']}", ylabel="Daily units")
    for horizon in config["forecast"]["cumulative_horizons"]:
        totals = sales.rolling(horizon, min_periods=horizon).sum().dropna()
        protection_rows.append({
            "regime": item["regime"], "SKU": item["item_id"], "days": horizon,
            "windows": len(totals), "zero windows": totals.eq(0).mean(),
            "median units": totals.quantile(0.50), "90th-percentile units": totals.quantile(0.90),
        })
plt.show()
display(pd.DataFrame(protection_rows).round(2))

### What would an always-zero forecast tell us?

I tried a forecast of zero for every date as a diagnostic. Compare it with yesterday's sales and the same weekday last week, on the same training dates with seven earlier observations. Exact-match rate rewards correct zeros, while mean absolute error exposes the units missed. The forecast and inventory notebooks will evaluate on separate dates. This training exercise only shows why one metric can be misleading and why the seven-day reference is worth testing.

In [ ]:
grouped_sales = train.groupby("item_id")["sales"]
actual = train["sales"]
yesterday = grouped_sales.shift(1)
last_week = grouped_sales.shift(7)
common = last_week.notna()
labels = train.loc[common, "item_id"].map(selected.set_index("item_id")["regime"])
diagnostics = []
for name, prediction in [("Always zero", pd.Series(0, index=train.index)), ("Yesterday", yesterday), ("Same weekday last week", last_week)]:
    errors = (actual.loc[common] - prediction.loc[common]).abs()
    diagnostics.append(
        pd.DataFrame({"regime": labels, "absolute error": errors, "exact match": errors.eq(0)})
        .groupby("regime", as_index=False)
        .agg(item_days=("absolute error", "size"), MAE=("absolute error", "mean"), exact_match_share=("exact match", "mean"))
        .assign(rule=name)
    )
display(pd.concat(diagnostics, ignore_index=True)[["regime", "rule", "item_days", "MAE", "exact_match_share"]].round(3))

**What surprised me in CA_1.** The zero rule also achieves a lower training MAE than either naïve rule for Intermittent items. When zero is the median daily outcome, this makes sense under absolute loss. If we used that zero point forecast directly as a stocking target, it would call for no stock even when occasional demand matters. I want the later evaluation to ask two different questions. How good is the forecast at the requested quantile, and what service did the policy deliver at the cost it incurred?

## 3. Change future observations and check the regime

This example gives one SKU four training days and four later days. Replacing every later sale with 100,000 units must reproduce the same training statistics and regime.

In [ ]:
fixture = pd.DataFrame({
    "store_id": "EXAMPLE",
    "item_id": "ONE",
    "date": pd.date_range("2020-01-01", periods=8),
    "sales": [3, 0, 3, 0, 1, 3, 4, 1],
})
start, end = "2020-01-01", "2020-01-04"
original = classify(fixture, start=start, end=end)
perturbed = fixture.copy()
perturbed.loc[perturbed["date"].gt(pd.Timestamp(end)), "sales"] = 100_000
pd.testing.assert_frame_equal(original, classify(perturbed, start=start, end=end))
display(original[["regime", "training_days", "positive_days", "active_zero_share", "positive_cv2"]])
print("PASS: later observations did not change any training regime statistic.")

## What we decided after exploring

| Question | Decision for the next stage | Still open |
|---|---|---|
| Is a zero interchangeable with a missing price? | Preserve zeros and null prices as separate facts | Which zeros reflect lost demand or listing changes |
| Are all SKUs alike? | Report results by training regime and across the store | Whether the cutoff or the chosen training span changes the conclusions |
| Which calendar and price signals help? | Try those observable at the origin and compare to the same baseline without them | Their out-of-sample contribution |
| Does a strong daily score improve stocking? | Evaluate the two protection-period forecasts and compare simulated cost and fill rate | Whether forecast gains survive the ordering policy and how much service they buy |

The April block has stayed outside these explorations. These choices can be challenged on the designated development, calibration and policy-validation dates before we open the final test.

The SKU regimes Parquet is the durable label table. The CSVs preserve the training profiles and threshold counts for later forecast and inventory comparisons.